In [1]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
import pathlib
import torch
import torch.nn as nn
from tqdm import tqdm
from torchmetrics.regression import (
    R2Score,
    MeanSquaredError,
    MeanAbsoluteError,
    PearsonCorrCoef,
)

from cpmp.model.transformer import CPMPGraphTransformer
from src.dataset import CPMPQM9DataModule

torch.set_float32_matmul_precision("high")

/home/pham/miniforge3/envs/cp3d/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class ARGS:
    def __init__(self, configs):
        for key, value in configs.items():
            setattr(self, key, value)


d_model = 64
h = 64
N = 6
N_dense = 2
slope = 0.16
drop = 0.1
lambda_attention = 0.1
lambda_distance = 0.6
aggregation = "dummy_node"
epochs = 7
max_patience = 100


configs = {
    "data_dir": pathlib.Path("data/cpmp/qm9"),
    "ff": "mmff",
    "ig": False,
    "wh": False,
    "split_seed": 0,
    "pdb": False,
    "amp": True,
    "log_dir": pathlib.Path("./results/cpmp/qm9"),
    "epochs": 30,
    "eval_interval": 1,
    "save_ckpt_path": pathlib.Path("./results/cpmp/qm9/new_env_test_egnn.pth"),
    "learning_rate": 0.01,
    "min_learning_rate": 1e-7,
    "batch_size": 512 * 4,
}

args = ARGS(configs)

In [4]:
loss_fn = nn.MSELoss(reduction="mean")
datamodule = CPMPQM9DataModule(
    split_seed=args.split_seed,
    data_dir=args.data_dir,
    batch_size=args.batch_size,
)
train_dataloader = datamodule.train_dataloader()
val_dataloader = datamodule.val_dataloader()

100%|██████████| 133885/133885 [00:11<00:00, 11899.48it/s]


In [5]:
d_atom = datamodule.d_atom
model_params = {
    "d_atom": d_atom,
    "d_model": d_model,
    "N": N,
    "h": h,
    "N_dense": N_dense,
    "lambda_attention": lambda_attention,
    "lambda_distance": lambda_distance,
    "leaky_relu_slope": slope,
    "dense_output_nonlinearity": "relu",
    "distance_matrix_kernel": "exp",
    "dropout": drop,
    "aggregation_type": aggregation,
}
model = CPMPGraphTransformer(**model_params)

In [6]:
device = torch.cuda.current_device()
model.to(device=device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.learning_rate,
    fused=True,
)

mae = MeanAbsoluteError()
rmse = MeanSquaredError(squared=False)
r2 = R2Score()
pearson = PearsonCorrCoef()

for epoch in range(args.epochs):
    loss_acc = torch.zeros((1,), device=device)
    model.train()
    for i, batch in tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        unit="batch",
        desc=f"Epoch {epoch}",
        leave=False,
    ):
        adjacency_matrix, node_features, distance_matrix, target = batch
        batch_mask = torch.sum(torch.abs(node_features), dim=-1) != 0
        adjacency_matrix = adjacency_matrix.to(device, non_blocking=True)
        node_features = node_features.to(device, non_blocking=True)
        distance_matrix = distance_matrix.to(device, non_blocking=True)
        batch_mask = batch_mask.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        pred = model(node_features, batch_mask, adjacency_matrix, distance_matrix, None)
        loss = loss_fn(pred.flatten(), target.flatten())
        loss_acc += loss.detach()
        loss.backward()
        optimizer.step()
        model.zero_grad(set_to_none=True)
    print(f"Epoch {epoch}")
    print(f"Loss {loss_acc.item() / len(train_dataloader):.4f}")

    model.eval()
    with torch.inference_mode():
        for i, batch in tqdm(
            enumerate(val_dataloader),
            total=len(val_dataloader),
            unit="batch",
            desc=f"Epoch {epoch}",
            leave=False,
        ):
            adjacency_matrix, node_features, distance_matrix, target = batch
            batch_mask = torch.sum(torch.abs(node_features), dim=-1) != 0
            adjacency_matrix = adjacency_matrix.to(device, non_blocking=True)
            node_features = node_features.to(device, non_blocking=True)
            distance_matrix = distance_matrix.to(device, non_blocking=True)
            batch_mask = batch_mask.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            pred = model(
                node_features, batch_mask, adjacency_matrix, distance_matrix, None
            )
            pred_flat, target_flat = (
                pred.detach().view(-1).float().cpu(),
                target.detach().view(-1).float().cpu(),
            )

            mae(pred_flat, target_flat)
            rmse(pred_flat, target_flat)
            r2(pred_flat, target_flat)
            pearson(pred_flat, target_flat)

        print(
            f"""
            validation MAE: {float(mae.compute()):.4f}
            validation RMSE: {float(rmse.compute()):.4f}
            validation R²: {float(r2.compute()):.4f}
            validation Pearson r: {float(pearson.compute()):.4f}
            """
        )

        mae.reset()
        rmse.reset()
        r2.reset()
        pearson.reset()

Epoch 0
Loss 1.3664



            validation MAE: 0.4462
            validation RMSE: 0.6093
            validation R²: -0.0311
            validation Pearson r: 0.2157
            


Epoch 1
Loss 0.3734



            validation MAE: 0.4505
            validation RMSE: 0.6143
            validation R²: -0.0481
            validation Pearson r: 0.2486
            


Epoch 2
Loss 0.3646


...


            validation MAE: 0.4174
            validation RMSE: 0.5550
            validation R²: 0.1446
            validation Pearson r: 0.3986
            


Epoch 28
Loss 0.3032



            validation MAE: 0.4143
            validation RMSE: 0.5526
            validation R²: 0.1519
            validation Pearson r: 0.4006
            


Epoch 29
Loss 0.3036



            validation MAE: 0.4321
            validation RMSE: 0.5672
            validation R²: 0.1067
            validation Pearson r: 0.4027
            
